In [ ]:
import numpy as np
import pandas as pd
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 1.21.6
Pandas version: 1.5.3


In [ ]:
import gensim
print(gensim.__version__)

4.1.2


In [ ]:
import time
import random
import os
import urllib.request
import zipfile
import pickle
from tqdm import tqdm
tqdm.pandas()  # <- 이걸 실행해야 progress_apply가 활성화됨

# 데이터 분석 라이브러리
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import re
from nltk.tokenize import word_tokenize, sent_tokenize
import gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

# 케라스 라이브러리
import tensorflow as tf
from tensorflow.keras import backend as K, layers, Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Multiply, Concatenate, Dropout, Dense, Dot, Add, Embedding, Conv1D, GlobalMaxPooling1D, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 데이터 분할 및 인코더
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# 성능 평가 라이브러리
from sklearn.metrics import mean_absolute_error, mean_squared_error

(데이터 분할 / 저장)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import ast
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Input, Dense, Flatten, Concatenate, Multiply, Dot, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
path = " "

In [ ]:
# pickle 파일에서 훈련 데이터와 테스트 데이터 불러오기
with open(f'{path}/office_train_dict.pkl', 'rb') as f:
    train_dict = pickle.load(f)

with open(f'{path}/office_test_dict.pkl', 'rb') as f:
    test_dict = pickle.load(f)

# 불러온 데이터 확인
train_user_bart = train_dict["train_user_bart"]
train_item_bart = train_dict["train_item_bart"]
train_user_emotions = train_dict["train_user_emotions"]
train_item_emotions = train_dict["train_item_emotions"]
train_y = train_dict["rating"]

test_user_bart = test_dict["test_user_bart"]
test_item_bart = test_dict["test_item_bart"]
test_user_emotions = test_dict["test_user_emotions"]
test_item_emotions = test_dict["test_item_emotions"]
test_y = test_dict["rating"]

In [ ]:
def GMU(x1, x2, units=256, name_prefix=''):
    # projection
    x1_proj = Dense(units, activation='tanh', name=f'{name_prefix}_proj1')(x1)
    x2_proj = Dense(units, activation='tanh', name=f'{name_prefix}_proj2')(x2)

    # gate
    z = Dense(units, activation='sigmoid', name=f'{name_prefix}_gate')(
        Concatenate()([x1, x2])
    )

    # gated combination
    out = tf.keras.layers.Multiply()([z, x1_proj]) + tf.keras.layers.Multiply()([(1 - z), x2_proj])
    return out

In [ ]:
def bart_projection(x, name_prefix='bart'):
    x = Dense(512, activation='relu', name=f'{name_prefix}_dense1')(x)
    x = Dense(256, activation='relu', name=f'{name_prefix}_dense2')(x)
    x = Dense(256, activation='tanh', name=f'{name_prefix}_dense3')(x)
    return x

In [ ]:
def ProposedModel(hidden_units=[128, 64, 32], dropout_rate=0.2):

    # input

    # 리뷰 텍스트 요약
    user_bart_input = Input(shape=(768,), dtype=tf.float32, name='user_review_input')
    item_bart_input = Input(shape=(768,), dtype=tf.float32, name='item_review_input')

    # 감정
    user_emotion_input = Input(shape=(7,), dtype=tf.float32, name='user_emotion_input')
    item_emotion_input = Input(shape=(7,), dtype=tf.float32, name='item_emotion_input')


    # BART 임베딩 projection
    user_bart_proj = bart_projection(user_bart_input, name_prefix='user_bart')
    item_bart_proj = bart_projection(item_bart_input, name_prefix='item_bart')

    # 감정 벡터 projection
    user_emotion_proj = Dense(256, activation='tanh', name='user_emotion_proj')(user_emotion_input)
    item_emotion_proj = Dense(256, activation='tanh', name='item_emotion_proj')(item_emotion_input)

    # user

    # user_bart + user_emotion concat -> user feature
    user_feature = GMU(user_bart_proj, user_emotion_proj, units=256, name_prefix='user')

    # item

    # item_bart + item_emtion concat -> item feature
    item_feature = GMU(item_bart_proj, item_emotion_proj, units=256, name_prefix='item')


    # user feature + item feature 결합
    x = Concatenate(name = 'user_item_feature')([user_feature, item_feature])


    for i, units in enumerate(hidden_units):
        x = Dense(units, activation='relu', name=f'dense_{i+1}')(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate, name=f'dropout_{i+1}')(x)


    output = Dense(1, activation='linear', name = 'output_layer')(x)


    model = Model(inputs=[user_bart_input, item_bart_input, user_emotion_input, item_emotion_input], outputs=output)

    return model

In [ ]:
proposed_model = ProposedModel()
adam = Adam(learning_rate=0.0002)
proposed_model.compile(optimizer=adam, loss='MSE', metrics=["mse", "mae"])

In [ ]:
proposed_model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_review_input (InputLayer)  [(None, 768)]       0           []                               
                                                                                                  
 item_review_input (InputLayer)  [(None, 768)]       0           []                               
                                                                                                  
 user_bart_dense1 (Dense)       (None, 512)          393728      ['user_review_input[0][0]']      
                                                                                                  
 item_bart_dense1 (Dense)       (None, 512)          393728      ['item_review_input[0][0]']      
                                                                                            

In [ ]:
# EarlyStopping 설정
early_stopping = EarlyStopping(
    monitor='val_loss',  # 검증 손실을 모니터링
    min_delta=0.001,  # 개선된 최소 값
    patience=5,  # 개선이 없으면 훈련을 중단할 에포크 수
    verbose=1,  # 훈련 중단 시 경고 메시지 출력
    mode='min',  # val_loss가 최소화되어야 하므로 'min'
    restore_best_weights=True  # 가장 좋은 성능의 모델 가중치를 복원
)

In [ ]:
# 모델 학습
history = proposed_model.fit(
    [train_user_bart, train_item_bart, train_user_emotions, train_item_emotions],
    train_y,
    validation_split=0.125,
    batch_size=128,
    callbacks=[early_stopping],
    epochs=50
)

Epoch 1/50
2046/2046 [==============================] - 18s 9ms/step - loss: 1.5399 - mse: 1.5399 - mae: 0.9532 - val_loss: 1.0742 - val_mse: 1.0742 - val_mae: 0.9207
Epoch 2/50
2046/2046 [==============================] - 18s 9ms/step - loss: 1.1717 - mse: 1.1717 - mae: 0.8301 - val_loss: 0.9441 - val_mse: 0.9441 - val_mae: 0.8467
Epoch 3/50
2046/2046 [==============================] - 18s 9ms/step - loss: 1.0808 - mse: 1.0808 - mae: 0.7899 - val_loss: 0.9613 - val_mse: 0.9613 - val_mae: 0.8607
Epoch 4/50
2046/2046 [==============================] - 18s 9ms/step - loss: 1.0175 - mse: 1.0175 - mae: 0.7598 - val_loss: 0.8253 - val_mse: 0.8253 - val_mae: 0.7621
Epoch 5/50
2046/2046 [==============================] - 18s 9ms/step - loss: 0.9475 - mse: 0.9475 - mae: 0.7254 - val_loss: 0.6954 - val_mse: 0.6954 - val_mae: 0.6412
Epoch 6/50
2046/2046 [==============================] - 17s 8ms/step - loss: 0.8933 - mse: 0.8933 - mae: 0.6978 - val_loss: 0.6880 - val_mse: 0.6880 - val_mae: 0.627

In [ ]:
# 성능 평가 (MAE, MSE, RMSE, MAPE)
y_pred = proposed_model.predict(
    [test_user_bart, test_item_bart, test_user_emotions, test_item_emotions]  # 테스트 데이터 입력
).flatten()  # 예측값을 1차원 배열로 변환

mae = mean_absolute_error(test_y, y_pred)
mse = mean_squared_error(test_y, y_pred)
rmse = np.sqrt(mse)
mape = 100 * mean_absolute_percentage_error(test_y, y_pred)

# 평가 결과 출력
print(f"MAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.3f}")

MAE: 0.559
MSE: 0.661
RMSE: 0.813
MAPE: 20.096
